## Train MLP

In [1]:
import sys
import pandas as pd
import numpy as np

sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from dim_reduction_utils import load_imbalanced_cryptic_and_regular_data, load_dataset_with_all_balanced_classes
from constants import CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET, IMG_OUTPUT_PATH, EMB_SPACES, CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET, EMBEDDINGS_PATH

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, matthews_corrcoef, accuracy_score, confusion_matrix
from sklearn.utils import shuffle

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


MODELS_PATH = f'{EMBEDDINGS_PATH}/../models'


In [2]:
DROPOUT = 0.3
LAYER_WIDTH = 1024

class EmmaClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=input_dim, out_features=LAYER_WIDTH)
        self.dropout1 = nn.Dropout(DROPOUT)

        self.layer_2 = nn.Linear(in_features=LAYER_WIDTH, out_features=LAYER_WIDTH)
        self.dropout2 = nn.Dropout(DROPOUT)

        self.layer_3 = nn.Linear(in_features=LAYER_WIDTH, out_features=LAYER_WIDTH)
        self.dropout3 = nn.Dropout(DROPOUT)

        self.layer_4 = nn.Linear(in_features=LAYER_WIDTH, out_features=3)

        self.relu = nn.ReLU()

    def forward(self, x):
      # Intersperse the ReLU activation function between layers
       return self.layer_4(self.dropout3(self.relu(self.layer_3(self.dropout2(self.relu(self.layer_2(self.dropout1(self.relu(self.layer_1(x))))))))))


class EmmaDataset(Dataset):
    def __init__(self, emma, emb_space_name):
        self.X = emma.emb[emb_space_name]['emb']
        self.y = np.column_stack((
            (emma.metadata['binding_site'] == 'BINDING').to_numpy().astype(int),
            (emma.metadata['binding_site'] == 'CRYPTIC-BINDING').to_numpy().astype(int),
            (emma.metadata['binding_site'] == 'NON-BINDING').to_numpy().astype(int)))

        self.X, self.y = shuffle(self.X, self.y, random_state=42)

        self.Xs = torch.tensor(self.X, dtype=torch.float32)
        self.Ys = torch.tensor(self.y, dtype=torch.int64)

    def __len__(self):
        assert len(self.Xs) == len(self.Ys)
        return len(self.Xs)

    def __getitem__(self, idx):
        x = self.Xs[idx]
        y = self.Ys[idx]
        return x, y

def compute_weights(y):
    counts = y.sum(axis=0)

    N = y.shape[0]
    C = y.shape[1]

    weights = N / (C * counts)
    return weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on device {device} ...')
torch.manual_seed(42)

Running on device cuda ...


In [3]:
def train(emb_space, epochs=30, batch_size=4096, balanced=False):
        
    if not balanced:
        train_emma = load_imbalanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET])
        test_emma = load_imbalanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET])
    else:
        train_emma = load_dataset_with_all_balanced_classes(load_train_subset=True)
        test_emma = load_dataset_with_all_balanced_classes(load_train_subset=False)

    train_dataset = EmmaDataset(train_emma, emb_space[0])
    test_dataset = EmmaDataset(test_emma, emb_space[0])

    X_test, y_test = test_dataset[:]

    X_test, y_test = X_test.to(device), y_test.to(device).float()
    train_losses = []
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    class FocalLoss(nn.Module):
        def __init__(self, alpha=None, gamma=2):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma

        def forward(self, logits, targets):
            ce_loss = nn.functional.cross_entropy(
                logits, targets, weight=self.alpha, reduction='none'
            )
            pt = torch.exp(-ce_loss)
            focal_loss = (1 - pt) ** self.gamma * ce_loss
            return focal_loss.mean()


    model = EmmaClassifier(input_dim=train_dataset.X.shape[1]).to(device)
    optimizer = torch.optim.AdamW(params=model.parameters(),
                                lr=0.00001)
    
    criterion = FocalLoss(alpha=torch.tensor([2, 7, 0.5]).float().to(device), gamma=1.5)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    for epoch in range(epochs):
        #
        # TEST
        #
        model.eval()
        with torch.inference_mode():

            test_logits = model(X_test).squeeze()
            test_probabilities = torch.softmax(test_logits, dim=1)

            test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
            y_true = torch.argmax(y_test, dim=1).cpu().numpy()
            y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

            cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
            print("Confusion matrix:")
            print(cm)
            per_class_acc = np.divide(
                cm.diagonal(),
                cm.sum(axis=1),
                out=np.zeros(cm.shape[0], dtype=float),
                where=cm.sum(axis=1) != 0
            )

            class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
            class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

            test_acc = accuracy_score(y_true, y_pred)
            test_mcc = matthews_corrcoef(y_true, y_pred)
            test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
            test_f1_macro = f1_score(y_true, y_pred, average='macro')
            test_f1_micro = f1_score(y_true, y_pred, average='micro')
            test_f1 = test_f1_weighted
            test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
            test_auprc_macro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
            test_auprc_micro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
            test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
            test_roc_auc_macro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
            test_roc_auc_micro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')

        #
        # TRAIN
        #
        model.train()
        batch_losses = []
        for x_batch, y_batch in train_dataloader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_logits = model(x_batch)

            y_batch = torch.argmax(y_batch, dim=1)
            loss = criterion(y_logits,
                            y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.cpu().detach().numpy())

        train_losses.append(sum(batch_losses) / len(batch_losses))

        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, MCC: {test_mcc:.4f}, weighted F1: {\
        test_f1:.4f}, micro F1: {test_f1_micro:.4f}, macro F1: {test_f1_macro:.4f}, weighted AUPRC: {test_auprc:.4f}, micro AUPRC: {test_auprc_micro:.4f}, macro AUPRC: {test_auprc_macro:.4f}, weighted ROC AUC: {test_roc_auc:.4f}, micro ROC AUC: {test_roc_auc_micro:.4f}, macro ROC AUC: {test_roc_auc_macro:.4f}")
        print(f"Class-wise accuracy:\n\t{class_acc_df}\n")
        print()

    print("Best model performance:")
    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
        test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
        test_f1_macro = f1_score(y_true, y_pred, average='macro')
        test_f1_micro = f1_score(y_true, y_pred, average='micro')
        test_f1 = test_f1_weighted
        print(test_probabilities.cpu().numpy().shape, y_test.cpu().numpy().shape)
        test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_auprc_macro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
        test_auprc_micro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
        test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_roc_auc_micro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
        test_roc_auc_macro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
    print(f"Test loss: {test_loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, MCC: {test_mcc:.4f}, weighted F1: {\
        test_f1:.4f}, micro F1: {test_f1_micro:.4f}, macro F1: {test_f1_macro:.4f}, weighted AUPRC: {test_auprc:.4f}, micro AUPRC: {\
        test_auprc_micro:.4f}, macro AUPRC: {test_auprc_macro:.4f}, weighted ROC AUC: {test_roc_auc:.4f}, micro ROC AUC: {test_roc_auc_micro:.4f}, macro ROC AUC: {test_roc_auc_macro:.4f}")

    # with open(f'{MODELS_PATH}/{emb_space[0]}_train2_torch_best_model.pth', 'wb') as f:
    #     torch.save(model, f)
    # import pickle
    # with open(f'{IMG_OUTPUT_PATH}/performance/train2-epochs={epochs},weighted-metrics/{emb_space[0]}_torch.pkl', 'wb') as f:
    #     pickle.dump({
    #         "test_acc": test_acc,
    #         "test_roc_auc_micro": test_roc_auc_micro,
    #         "test_roc_auc_macro": test_auprc_macro,
    #         "test_mcc": test_mcc,
    #         "test_f1_weighted": test_f1_weighted,
    #         "test_f1_macro": test_f1_macro,
    #         "test_f1_micro": test_f1_micro,
    #         "test_auprc_macro": test_auprc_micro,
    #         "test_auprc_micro": test_auprc_micro,
    #         "class_acc_df": class_acc_df,
    #         "cm": cm
    #     }, f)


In [9]:
for emb_space in EMB_SPACES:
    train(emb_space, epochs=10)

2042764 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Confusion matrix:
[[     0      6  74811]
 [     0      1   3157]
 [     1     63 734047]]
(812086, 3) (812086, 3)
Epoch: 0 | Loss: 0.24224, Accuracy: 90.39% | Test loss: 0.33342, MCC: 0.0002, weighted F1: 0.8584, micro F1: 0.9039, macro F1: 0.3167, weighted AUPRC: 0.8262, micro AUPRC: 0.8607, macro AUPRC: 0.3358, weighted ROC AUC: 0.5105, micro ROC AUC: 0.9175, macro ROC AUC: 0.5211
Class-wise accuracy:
	             class  accuracy
0          BINDING  0.000000
1  CRYPTIC-BINDING  0.000317
2      NON-BINDING  0.999913


Confusion matrix:
[[ 59468      0  15349]
 [  2644      0    514]
 [276647      0 457464]]
(812086, 3) (812086, 3)

In [ ]:
for emb_space in EMB_SPACES:
    with open(f'{MODELS_PATH}/{emb_space[0]}_train2_torch_best_model.pth', 'rb') as f:
        model = torch.load(f, map_location=device, weights_only=False)
    test_emma = load_imbalanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET])
    test_dataset = EmmaDataset(test_emma, emb_space[0])

    X_test, y_test = test_dataset[:]
    X_test, y_test = X_test.to(device), y_test.to(device).float()

    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
        test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
        test_f1_macro = f1_score(y_true, y_pred, average='macro')
        test_f1_micro = f1_score(y_true, y_pred, average='micro')
        test_f1 = test_f1_weighted
        test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
    print(f"Accuracy: {(test_acc * 100):.2f}%, AUC: {test_roc_auc:.4f}, MCC: {test_mcc:.4f}, F1: {test_f1:.4f}, AUPRC: {test_auprc:.4f}")


812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Confusion matrix:
[[ 26487  26220  22110]
 [  1791    720    647]
 [ 84710 112258 537143]]
Accuracy: 69.49%, AUC: 0.7825, MCC: 0.2021, F1: 0.7765, AUPRC: 0.8949
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Confusion matrix:
[[ 40681  13022  21114]
 [  2097    479    582]
 [142089  27616 564406]]
Accuracy: 74.57%, AUC: 0.8154, MCC: 0.2650, F1: 0.8019, AUPRC: 0.9034
795561 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Confusion matrix:
[[ 34474  16892  22176]
 [  1807    689    603]
 [126586  52122 540212]]
Accuracy: 72.32%, AUC: 0.7892, MCC: 0.2300, F1:

## Train on balanced dataset

In [ ]:
def train_balanced(emb_space, epochs=30, batch_size=4096):
    
    train_emma = load_dataset_with_all_balanced_classes(load_train_subset=True)
    test_emma = load_dataset_with_all_balanced_classes(load_train_subset=False)
    
    train_dataset = EmmaDataset(train_emma, emb_space[0])
    test_dataset = EmmaDataset(test_emma, emb_space[0])
    print(f'Computing for {emb_space[0]} ...')
    X_test, y_test = test_dataset[:]

    X_test, y_test = X_test.to(device), y_test.to(device).float()
    train_losses = []
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    class FocalLoss(nn.Module):
        def __init__(self, alpha=None, gamma=2):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma

        def forward(self, logits, targets):
            ce_loss = nn.functional.cross_entropy(
                logits, targets, weight=self.alpha, reduction='none'
            )
            pt = torch.exp(-ce_loss)
            focal_loss = (1 - pt) ** self.gamma * ce_loss
            return focal_loss.mean()


    model = EmmaClassifier(input_dim=train_dataset.X.shape[1]).to(device)
    optimizer = torch.optim.AdamW(params=model.parameters(),
                                lr=0.00001)
    
    criterion = FocalLoss(gamma=1.5) # ,alpha=torch.tensor([2, 7, 0.5]).float().to(device))
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    for epoch in range(epochs):
        #
        # TEST
        #
        model.eval()
        with torch.inference_mode():

            test_logits = model(X_test).squeeze()
            test_probabilities = torch.softmax(test_logits, dim=1)

            test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
            y_true = torch.argmax(y_test, dim=1).cpu().numpy()
            y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

            cm = confusion_matrix(y_true, y_pred) #, labels=np.arange(y_test.shape[1]))
            print("Confusion matrix:")
            print(cm)
            per_class_acc = np.divide(
                cm.diagonal(),
                cm.sum(axis=1),
                out=np.zeros(cm.shape[0], dtype=float),
                where=cm.sum(axis=1) != 0
            )

            class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
            class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

            test_acc = accuracy_score(y_true, y_pred)
            test_mcc = matthews_corrcoef(y_true, y_pred)
            test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
            test_f1_macro = f1_score(y_true, y_pred, average='macro')
            test_f1_micro = f1_score(y_true, y_pred, average='micro')
            test_f1 = test_f1_weighted
            test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
            test_auprc_macro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
            test_auprc_micro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
            test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
            test_roc_auc_macro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
            test_roc_auc_micro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')

        #
        # TRAIN
        #
        model.train()
        batch_losses = []
        for x_batch, y_batch in train_dataloader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_logits = model(x_batch)

            y_batch = torch.argmax(y_batch, dim=1)
            loss = criterion(y_logits,
                            y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.cpu().detach().numpy())

        train_losses.append(sum(batch_losses) / len(batch_losses))

        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, MCC: {test_mcc:.4f}, weighted F1: {\
        test_f1:.4f}, micro F1: {test_f1_micro:.4f}, macro F1: {test_f1_macro:.4f}, weighted AUPRC: {test_auprc:.4f}, micro AUPRC: {test_auprc_micro:.4f}, macro AUPRC: {test_auprc_macro:.4f}, weighted ROC AUC: {test_roc_auc:.4f}, micro ROC AUC: {test_roc_auc_micro:.4f}, macro ROC AUC: {test_roc_auc_macro:.4f}")
        print(f"Class-wise accuracy:\n\t{class_acc_df}\n")
        print()

    print("Best model performance:")
    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        test_loss = criterion(test_logits, torch.argmax(y_test, dim=1))
        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred) #, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
        test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
        test_f1_macro = f1_score(y_true, y_pred, average='macro')
        test_f1_micro = f1_score(y_true, y_pred, average='micro')
        test_f1 = test_f1_weighted
        test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_auprc_macro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
        test_auprc_micro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
        test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_roc_auc_micro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
        test_roc_auc_macro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
    print(f"Test loss: {test_loss:.5f}, Accuracy: {(test_acc * 100):.2f}% | Test loss: {test_loss:.5f}, MCC: {test_mcc:.4f}, weighted F1: {\
        test_f1:.4f}, micro F1: {test_f1_micro:.4f}, macro F1: {test_f1_macro:.4f}, weighted AUPRC: {test_auprc:.4f}, micro AUPRC: {\
        test_auprc_micro:.4f}, macro AUPRC: {test_auprc_macro:.4f}, weighted ROC AUC: {test_roc_auc:.4f}, micro ROC AUC: {test_roc_auc_micro:.4f}, macro ROC AUC: {test_roc_auc_macro:.4f}")

    with open(f'{MODELS_PATH}/{emb_space[0]}_train,balanced,e={epochs}.pth', 'wb') as f:
        torch.save(model, f)
    import pickle
    with open(f'{IMG_OUTPUT_PATH}/performance/train,balanced,e={epochs}/{emb_space[0]}_torch.pkl', 'wb') as f:
        pickle.dump({
            "test_acc": test_acc,
            "test_roc_auc_micro": test_roc_auc_micro,
            "test_roc_auc_macro": test_roc_auc_macro,
            "test_roc_auc_weighted": test_roc_auc,
            "test_mcc": test_mcc,
            "test_f1_weighted": test_f1_weighted,
            "test_f1_macro": test_f1_macro,
            "test_f1_micro": test_f1_micro,
            "test_auprc_macro": test_auprc_macro,
            "test_auprc_micro": test_auprc_micro,
            "test_auprc_weighted": test_auprc,
            "class_acc_df": class_acc_df,
            "cm": cm
        }, f)

for emb_space in EMB_SPACES:
    train_balanced(emb_space, epochs=10)

38898 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.
9336 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.
Computing for ESM2 ...
Confusion matrix:
[[  17 1871 1250]
 [  14 1778 1307]
 [  16 1176 1907]]
Epoch: 0 | Loss: 0.59470, Accuracy: 39.65% | Test loss: 0.59767, 

## Run on imbalanced test set
Take the model trained on balanced train set and run it on imbalanced test set.

In [4]:
for emb_space in EMB_SPACES:
    with open(f'{MODELS_PATH}/{emb_space[0]}_train,balanced,e=10.pth', 'rb') as f:
        model = torch.load(f, map_location=device, weights_only=False)
    test_emma = load_imbalanced_cryptic_and_regular_data(emb_space, [CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET])
    test_dataset = EmmaDataset(test_emma, emb_space[0])

    X_test, y_test = test_dataset[:]
    X_test, y_test = X_test.to(device), y_test.to(device).float()

    model.eval()
    with torch.inference_mode():

        test_logits = model(X_test).squeeze()
        test_probabilities = torch.softmax(test_logits, dim=1)

        y_true = torch.argmax(y_test, dim=1).cpu().numpy()
        y_pred = torch.argmax(test_probabilities, dim=1).cpu().numpy()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(y_test.shape[1]))
        print("Confusion matrix:")
        print(cm)
        per_class_acc = np.divide(
            cm.diagonal(),
            cm.sum(axis=1),
            out=np.zeros(cm.shape[0], dtype=float),
            where=cm.sum(axis=1) != 0
        )

        class_names = pd.get_dummies(test_emma.metadata["binding_site"], dummy_na=False).columns.tolist()
        class_acc_df = pd.DataFrame({"class": class_names, "accuracy": per_class_acc})

        test_acc = accuracy_score(y_true, y_pred)
        test_mcc = matthews_corrcoef(y_true, y_pred)
        test_f1_weighted = f1_score(y_true, y_pred, average='weighted')
        test_f1_macro = f1_score(y_true, y_pred, average='macro')
        test_f1_micro = f1_score(y_true, y_pred, average='micro')
        test_f1 = test_f1_weighted
        test_auprc = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_auprc_macro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
        test_auprc_micro = average_precision_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
        test_roc_auc = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='weighted')
        test_roc_auc_micro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='micro')
        test_roc_auc_macro = roc_auc_score(y_test.cpu().numpy(), test_probabilities.cpu().numpy(), average='macro')
    print(f"Accuracy: {(test_acc * 100):.2f}%, MCC: {test_mcc:.4f}, weighted F1: {\
        test_f1:.4f}, micro F1: {test_f1_micro:.4f}, macro F1: {test_f1_macro:.4f}, weighted AUPRC: {test_auprc:.4f}, micro AUPRC: {\
        test_auprc_micro:.4f}, macro AUPRC: {test_auprc_macro:.4f}, weighted ROC AUC: {test_roc_auc:.4f}, micro ROC AUC: {test_roc_auc_micro:.4f}, macro ROC AUC: {test_roc_auc_macro:.4f}")


812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Confusion matrix:
[[  6261  48627  19929]
 [   531   1699    928]
 [ 72453 249115 412543]]
Accuracy: 51.78%, MCC: 0.0791, weighted F1: 0.6464, micro F1: 0.5178, macro F1: 0.2664, weighted AUPRC: 0.8798, micro AUPRC: 0.5014, macro AUPRC: 0.3905, weighted ROC AUC: 0.7085, micro ROC AUC: 0.5793, macro ROC AUC: 0.6874
812086 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Confusion matrix:
[[ 34046  26672  14099]
 [  1437   1234    487]
 [203293 134319 396499]]
Accuracy: 53.17%, MCC: 0.1409, weighted F1: 0.6460, micro F1: 0.5317, macro F1: 0.3082, weighted AUPRC: 0.8950, micro AUPRC: 0.4942, macro AUPRC: 0.4184, weighted ROC AUC: 0.7649, micro ROC AUC: 0.5841, macro ROC AUC: 0.7651
795561 sam